<a href="https://colab.research.google.com/github/PARIMIANUDHEER04/PyTorch/blob/PyTorch/CNN_Architectures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **LeNet**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:06<00:00, 28.3MB/s]


In [4]:
class Lenet5(nn.Module):
  def __init__(self):
    super(Lenet5,self).__init__()
    self.conv1=nn.Conv2d(3,6,5)
    self.pool=nn.AvgPool2d(2,2)
    self.conv2=nn.Conv2d(6,16,5)
    self.fc1=nn.Linear(16*5*5,120)
    self.fc2=nn.Linear(120,84)
    self.fc3=nn.Linear(84,10)
  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=self.pool(x)
    x=F.relu(self.conv2(x))
    x=self.pool(x)
    x=x.view(-1, 16 * 5 * 5)
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    x=self.fc3(x)
    return x




In [5]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate model, loss, and optimizer
net = Lenet5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

In [6]:
for epoch in range(10):  # run for 10 epochs
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader, 0):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()              # reset gradients
        outputs = net(inputs)              # forward
        loss = criterion(outputs, labels)  # compute loss
        loss.backward()                    # backward
        optimizer.step()                   # update weights

        running_loss += loss.item()
        if i % 100 == 99:  # print every 100 mini-batches
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

[Epoch 1, Batch 100] loss: 2.091
[Epoch 1, Batch 200] loss: 1.861
[Epoch 1, Batch 300] loss: 1.773
[Epoch 2, Batch 100] loss: 1.656
[Epoch 2, Batch 200] loss: 1.616
[Epoch 2, Batch 300] loss: 1.580
[Epoch 3, Batch 100] loss: 1.536
[Epoch 3, Batch 200] loss: 1.531
[Epoch 3, Batch 300] loss: 1.522
[Epoch 4, Batch 100] loss: 1.473
[Epoch 4, Batch 200] loss: 1.479
[Epoch 4, Batch 300] loss: 1.444
[Epoch 5, Batch 100] loss: 1.448
[Epoch 5, Batch 200] loss: 1.437
[Epoch 5, Batch 300] loss: 1.415
[Epoch 6, Batch 100] loss: 1.402
[Epoch 6, Batch 200] loss: 1.380
[Epoch 6, Batch 300] loss: 1.389
[Epoch 7, Batch 100] loss: 1.378
[Epoch 7, Batch 200] loss: 1.367
[Epoch 7, Batch 300] loss: 1.346
[Epoch 8, Batch 100] loss: 1.346
[Epoch 8, Batch 200] loss: 1.331
[Epoch 8, Batch 300] loss: 1.323
[Epoch 9, Batch 100] loss: 1.326
[Epoch 9, Batch 200] loss: 1.312
[Epoch 9, Batch 300] loss: 1.311
[Epoch 10, Batch 100] loss: 1.286
[Epoch 10, Batch 200] loss: 1.280
[Epoch 10, Batch 300] loss: 1.286


In [7]:
import torch

# Make sure your model is in evaluation mode
net.eval()  # important: disables dropout, batchnorm, etc.

correct = 0
total = 0

# No need to compute gradients during evaluation
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)  # send to GPU if available

        # Forward pass
        outputs = net(inputs)

        # Get predicted class (index of max logit)
        _, predicted = torch.max(outputs, 1)

        # Update total and correct
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Compute accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')


Test Accuracy: 53.45%


## **AlexNet**

In [8]:
class AlexNet(nn.Module):
  def __init__(self,num_classes):
    super(AlexNet,self).__init__()
    self.conv1=nn.Conv2d(3,96,kernel_size=11,stride=4,padding=2)
    self.pool=nn.MaxPool2d(3,2)
    self.conv2=nn.Conv2d(96,256,5,1,2)
    self.conv3=nn.Conv2d(256,384,3,1,1)
    self.conv4=nn.Conv2d(384,384,3,1,1)
    self.conv5=nn.Conv2d(384,256,3,1,1)
    self.fc1 = nn.Linear(256 * 6 * 6, 4096)
    self.fc2 = nn.Linear(4096, 4096)
    self.fc3 = nn.Linear(4096, num_classes)
  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=self.pool(x)
    x=F.relu(self.conv2(x))
    x=self.pool(x)
    x=F.relu(self.conv3(x))
    x=F.relu(self.conv4(x))
    x=F.relu(self.conv5(x))
    x=self.pool(x)
    x=x.view(-1, 256 * 6 * 6)
    x=F.relu(self.fc1(x))
    x=F.dropout(x,0.5)
    x=F.relu(self.fc2(x))
    x=F.dropout(x,0.5)
    x=self.fc3(x)
    return x




In [9]:


class VGG16(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG16, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [10]:
class vgg19(nn.Module):
  def __init__(self):
    super(vgg19, self).__init__()
    self.block1=nn.Sequential(
        nn.Conv2d(3,64,3,1,1),
        nn.ReLU(),
        nn.Conv2d(64,64,3,1,1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )
    self.block2=nn.Sequential(
        nn.Conv2d(64,128,3,1,1),
        nn.ReLU(),
        nn.Conv2d(128,128,3,1,1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )
    self.block3=nn.Sequential(
        nn.Conv2d(128,256,3,1,1),
        nn.ReLU(),
        nn.Conv2d(256,256,3,1,1),
        nn.ReLU(),
        nn.Conv2d(256,256,3,1,1),
        nn.ReLU(),
        nn.Conv2d(256,256,3,1,1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )

    self.block4=nn.Sequential(
        nn.Conv2d(256,512,3,1,1),
        nn.ReLU(),
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )

    self.block5=nn.Sequential(
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.Conv2d(512,512,3,1,1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )

    self.classifier=nn.Sequential(
        nn.Linear(512*7*7,4096),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(4096,4096),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(4096,1000)
    )
  def forward(self,x):
    x=self.block1(x)
    x=self.block2(x)
    x=self.block3(x)
    x=self.block4(x)
    x=self.block5(x)
    x=torch.flatten(x,1)
    x=self.classifier(x)
    return x



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = vgg19().to(device)

# --- Data ---
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# --- Loss & Optimizer ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training ---
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch + 1}/{epochs}] Loss: {running_loss/len(trainloader):.4f}")

print("Training Completed!")


In [2]:
from torch.nn.modules import padding
class ResidualBlock(nn.Module):
  def __init__(self,inp,out,stride=1):
    super(ResidualBlock,self).__init__()
    self.conv1=nn.Conv2d(inp,out,kernel_size=3,stride=stride,padding=1)
    self.bn1=nn.BatchNorm2d(out)
    self.conv2=nn.Conv2d(out,out,kernel_size=3,padding=1)
    self.bn2=nn.BatchNorm2d(out)
    self.relu=nn.ReLU()
    self.shortcut=nn.Sequential()
    if stride!=1 or inp!=out:
      self.shortcut=nn.Sequential(
          nn.Conv2d(inp,out,1,stride),
          nn.BatchNorm2d(out)
      )
  def forward(self,x):
    out=self.relu(self.bn1(self.conv1(x)))
    out=self.bn2(self.conv2(out))
    out+=self.shortcut(x)
    return self.relu(out)
class resnet(nn.Module):
  def __init__(self):
    super(resnet,self).__init__()
    self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
    self.bn1 = nn.BatchNorm2d(64)
    self.relu = nn.ReLU()
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    self.layer1=nn.Sequential(
        ResidualBlock(64,64),
        ResidualBlock(64,64)
    )
    self.layer2=nn.Sequential(
        ResidualBlock(64,128,stride=2),
        ResidualBlock(128,128)
    )
    self.layer3=nn.Sequential(
        ResidualBlock(128,256,stride=2),
        ResidualBlock(256,256)
    )
    self.layer4=nn.Sequential(
        ResidualBlock(256,512,stride=2),
        ResidualBlock(512,512)
    )
    self.gvp=nn.AdaptiveAvgPool2d((1,1))
    self.fc=nn.Linear(512,10)
  def forward(self, x):
      x = self.relu(self.bn1(self.conv1(x)))
      x = self.maxpool(x)

      x = self.layer1(x)
      x = self.layer2(x)
      x = self.layer3(x)
      x = self.layer4(x)

      x = self.gvp(x)
      x = x.view(x.size(0), -1)
      return self.fc(x)






In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = resnet().to(device)

# --- Data ---
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# --- Loss & Optimizer ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training ---
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch + 1}/{epochs}] Loss: {running_loss/len(trainloader):.4f}")

print("Training Completed!")


100%|██████████| 170M/170M [00:04<00:00, 40.7MB/s]
